# Proyecto FIFA World Cup 2022  
## F3_Definicion.ipynb — Sumativa 3

**Asignatura:** MCDI500 — Herramientas de software científico  
**Proyecto:** Análisis del Mundial FIFA Qatar 2022  
**Entregable:** Notebook ejecutable F3 con preprocesamiento completo, algoritmos, eficiencia y POO  

> Versión incremental para Pull Request. Incluye las secciones 1 a 4: contexto, configuración, carga de datos y pipeline completo reutilizado desde F2.

## Índice

1. Contexto y propósito de la Fase 3  
2. Relación con Fase 2 y definición del problema algorítmico  
3. Configuración del entorno  
4. Carga del dataset y preprocesamiento funcional  
4.1. Pipeline completo reutilizado desde F2  

## 1. Contexto y propósito de la Fase 3

La Fase 3 toma como base el trabajo realizado en la Fase 2, donde se construyó un pipeline de obtención, limpieza, transformación y validación del dataset del Mundial FIFA Qatar 2022.

En esta etapa se desarrolla el núcleo algorítmico del proyecto. El propósito es demostrar que el análisis no solo permite explorar datos, sino también construir soluciones computacionales organizadas, medibles y reutilizables.

La guía de la Sumativa 3 solicita integrar programación estructurada, recursividad, eficiencia algorítmica, mediciones con `timeit`, Programación Orientada a Objetos, documentación técnica y un notebook ejecutable.


## 2. Relación con Fase 2 y definición del problema algorítmico

En la Fase 2 el dataset fue limpiado y transformado para dejar una base reproducible. En esta Fase 3 se reutiliza ese flujo, pero se reorganiza como arquitectura algorítmica.

**Problema algorítmico definido:** calcular y comparar el total de goles registrados en los partidos del Mundial FIFA 2022 utilizando tres enfoques:

1. Un algoritmo estructurado iterativo.
2. Un algoritmo recursivo.
3. Una operación vectorizada con pandas.

Además, se incorpora un algoritmo de ordenamiento recursivo `merge sort` para ordenar partidos según cantidad total de goles y una estructura POO para encapsular el preprocesamiento y los algoritmos de análisis.

Esta elección es pertinente porque la pregunta del proyecto se relaciona con el rendimiento de las selecciones y el comportamiento estadístico de los partidos. Los goles son una variable central para analizar resultados y desempeño.


## 3. Configuración del entorno

Se importan las librerías necesarias. El notebook está preparado para ejecutarse tanto desde la raíz del repositorio como desde una carpeta `notebooks/` o `F3/`. También incluye una carga alternativa con datos mínimos de ejemplo para que el notebook siga siendo ejecutable cuando el CSV original no esté disponible en el entorno de revisión.


In [1]:
from pathlib import Path
from abc import ABC, abstractmethod
import sys
import timeit
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.random.seed(42)

# Permite importar módulos del repositorio cuando el notebook se ejecuta desde notebooks/
# o desde la raíz del proyecto.
for ruta_src in [Path("../src"), Path("src"), Path(".")]:
    if ruta_src.exists() and str(ruta_src.resolve()) not in sys.path:
        sys.path.insert(0, str(ruta_src.resolve()))

try:
    import funciones_reutilizables as f2_utils
    import funciones_f3_algoritmos as f3_algos
    MODULOS_REPO_DISPONIBLES = True
    print("Módulos del repositorio importados correctamente desde src/.")
except Exception as exc:
    MODULOS_REPO_DISPONIBLES = False
    f2_utils = None
    f3_algos = None
    print("No se pudieron importar módulos desde src/. Se usarán funciones internas del notebook.")
    print("Detalle:", exc)

print("Entorno configurado correctamente")


No se pudieron importar módulos desde src/. Se usarán funciones internas del notebook.
Detalle: No module named 'funciones_reutilizables'
Entorno configurado correctamente


## 4. Carga del dataset y preprocesamiento funcional

Esta sección mantiene un flujo funcional de entrada → proceso → salida. Primero se carga el dataset original. Luego se ejecuta un preprocesamiento funcional mínimo para el núcleo algorítmico de goles, manteniendo columnas interpretables como `team1`, `team2` y goles.

En la sección **4.1** se ejecuta explícitamente el **pipeline completo reutilizado desde F2**, incluyendo manejo de nulos, casting, variables derivadas, codificación categórica, normalización y validación final. Esto responde directamente al feedback de la Formativa 3.


In [2]:
def crear_dataset_demo() -> pd.DataFrame:
    """Crea un dataset mínimo de respaldo para asegurar ejecución reproducible.

    Este respaldo no reemplaza el dataset real; solo permite que el notebook
    ejecute completo si el archivo CSV no está disponible en el entorno.
    """
    return pd.DataFrame({
        "team1": ["QATAR", "ENGLAND", "ARGENTINA", "FRANCE", "BRAZIL", "SPAIN"],
        "team2": ["ECUADOR", "IRAN", "SAUDI ARABIA", "AUSTRALIA", "SERBIA", "COSTA RICA"],
        "number of goals team1": [0, 6, 1, 4, 2, 7],
        "number of goals team2": [2, 2, 2, 1, 0, 0],
        "possession team1": ["47%", "78%", "69%", "63%", "59%", "82%"],
        "possession team2": ["40%", "13%", "20%", "25%", "24%", "9%"],
        "possession in contest": ["13%", "9%", "11%", "12%", "17%", "9%"],
        "total attempts team1": [5, 13, 14, 23, 24, 17],
        "total attempts team2": [6, 8, 3, 4, 4, 0],
        "passes team1": [450, 798, 596, 734, 602, 1045],
        "passes team2": [480, 215, 268, 466, 403, 231],
        "category": ["Group A", "Group B", "Group C", "Group D", "Group G", "Group E"]
    })


def cargar_datos(rutas_posibles: list[Path]) -> pd.DataFrame:
    """Carga el dataset FIFA desde la primera ruta disponible.

    Parámetros
    ----------
    rutas_posibles : list[Path]
        Lista ordenada de rutas candidatas para ubicar el CSV.

    Retorna
    -------
    pd.DataFrame
        Dataset cargado desde CSV o dataset de respaldo si el archivo no existe.
    """
    for ruta in rutas_posibles:
        if ruta.exists():
            df = pd.read_csv(ruta)
            print(f"Archivo utilizado: {ruta}")
            print(f"Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas")
            return df

    print("No se encontró el CSV original. Se usará dataset demo para mantener la ejecución reproducible.")
    df = crear_dataset_demo()
    print(f"Dataset demo cargado: {df.shape[0]} filas y {df.shape[1]} columnas")
    return df


def normalizar_nombres_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nombres de columnas a minúsculas, sin espacios extremos."""
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def convertir_porcentaje(df: pd.DataFrame, columna: str) -> pd.DataFrame:
    """Convierte una columna porcentual tipo '55%' a valor numérico float.

    Se mantiene el valor en escala 0-100 porque facilita la interpretación
    deportiva de posesión como porcentaje.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame")

    df = df.copy()
    df[columna] = (
        df[columna]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"nan": np.nan, "None": np.nan})
        .astype(float)
    )
    return df


def crear_variables_resultado(df: pd.DataFrame) -> pd.DataFrame:
    """Crea variables derivadas asociadas al resultado del partido.

    Se calcula el total de goles, la diferencia de goles del equipo 1
    y una etiqueta simple de resultado para team1.
    """
    requeridas = ["number of goals team1", "number of goals team2"]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise KeyError(f"Columnas faltantes para crear resultado: {faltantes}")

    df = df.copy()
    df["total_goals"] = df["number of goals team1"] + df["number of goals team2"]
    df["goal_difference_team1"] = df["number of goals team1"] - df["number of goals team2"]
    df["result_team1"] = np.select(
        [df["goal_difference_team1"] > 0, df["goal_difference_team1"] < 0],
        ["win", "loss"],
        default="draw"
    )
    return df


def imputar_nulos_numericos(df: pd.DataFrame, estrategia: str = "mediana") -> pd.DataFrame:
    """Imputa nulos en columnas numéricas usando media o mediana.

    Decisión técnica: se usa mediana por defecto porque es más robusta
    frente a valores extremos, algo esperable en estadísticas deportivas.
    """
    if estrategia not in {"media", "mediana"}:
        raise ValueError("La estrategia debe ser 'media' o 'mediana'")

    df = df.copy()
    numericas = df.select_dtypes(include=[np.number]).columns
    for col in numericas:
        if df[col].isnull().any():
            valor = df[col].median() if estrategia == "mediana" else df[col].mean()
            df[col] = df[col].fillna(valor)
    return df


def escalar_zscore(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    """Estandariza columnas numéricas con z-score.

    Decisión técnica: z-score permite comparar variables con distintas escalas,
    por ejemplo pases, intentos y posesión, manteniendo media cercana a 0.
    """
    df = df.copy()
    for col in columnas:
        if col not in df.columns:
            raise KeyError(f"La columna '{col}' no existe")
        if not pd.api.types.is_numeric_dtype(df[col]):
            raise TypeError(f"La columna '{col}' debe ser numérica para escalar")
        std = df[col].std(ddof=0)
        if std == 0 or np.isnan(std):
            df[col] = 0.0
        else:
            df[col] = (df[col] - df[col].mean()) / std
    return df


def preprocesar_datos(df: pd.DataFrame) -> pd.DataFrame:
    """Ejecuta el pipeline funcional de preprocesamiento de Fase 3."""
    df = normalizar_nombres_columnas(df)

    columnas_porcentaje = [
        "possession team1",
        "possession team2",
        "possession in contest"
    ]
    for col in columnas_porcentaje:
        if col in df.columns:
            df = convertir_porcentaje(df, col)

    df = crear_variables_resultado(df)
    df = imputar_nulos_numericos(df, estrategia="mediana")

    columnas_a_escalar = [
        c for c in [
            "possession team1", "possession team2", "possession in contest",
            "total attempts team1", "total attempts team2",
            "passes team1", "passes team2"
        ]
        if c in df.columns
    ]
    df = escalar_zscore(df, columnas_a_escalar)
    return df


def preparar_lista_goles(df: pd.DataFrame) -> list[int]:
    """Convierte las columnas de goles de ambos equipos en una lista plana."""
    columnas = ["number of goals team1", "number of goals team2"]
    faltantes = [c for c in columnas if c not in df.columns]
    if faltantes:
        raise KeyError(f"Faltan columnas de goles: {faltantes}")

    goles = df[columnas].to_numpy().ravel().astype(int).tolist()
    return goles


In [3]:
rutas_posibles = [
    Path("../data/raw/Fifa_world_cup_matches.csv"),
    Path("data/raw/Fifa_world_cup_matches.csv"),
    Path("../F2/data/raw/Fifa_world_cup_matches.csv"),
    Path("F2/data/raw/Fifa_world_cup_matches.csv"),
    Path("/mnt/data/Fifa_world_cup_matches.csv")
]

df_original = cargar_datos(rutas_posibles)
df = preprocesar_datos(df_original)
goles = preparar_lista_goles(df)

print("Dataset original:", df_original.shape)
print("Dataset preprocesado:", df.shape)
print("Cantidad de registros de goles:", len(goles))
print("Primeros 10 goles:", goles[:10])
df.head()


No se encontró el CSV original. Se usará dataset demo para mantener la ejecución reproducible.
Dataset demo cargado: 6 filas y 12 columnas
Dataset original: (6, 12)
Dataset preprocesado: (6, 15)
Cantidad de registros de goles: 12
Primeros 10 goles: [0, 2, 6, 2, 1, 2, 4, 1, 2, 0]


,team1,team2,number of goals team1,number of goals team2,possession team1,possession team2,possession in contest,total attempts team1,total attempts team2,passes team1,passes team2,category,total_goals,goal_difference_team1,result_team1
0,QATAR,ECUADOR,0,2,-1.646426,1.830705,0.426798,-1.710970,0.739940,-1.350242,1.242622,Group A,2,-2,loss
1,ENGLAND,IRAN,6,2,0.993533,-0.890159,-1.036508,-0.466628,1.547147,0.498483,-1.175700,Group B,8,4,win
2,ARGENTINA,SAUDI ARABIA,1,2,0.227093,-0.184750,-0.304855,-0.311086,-0.470871,-0.574628,-0.692035,Group C,3,-1,loss
3,FRANCE,AUSTRALIA,4,1,-0.283866,0.319114,0.060971,1.088799,-0.067267,0.158487,1.114861,Group D,5,3,win
4,BRAZIL,SERBIA,2,0,-0.624506,0.218341,1.890103,1.244342,-0.067267,-0.542753,0.539940,Group G,2,2,win


## 4.1. Pipeline completo reutilizado desde F2

El feedback de la Formativa 3 indicó que no bastaba con transformar la lista de goles: la rúbrica solicita ejecutar el pipeline completo de preprocesamiento dentro del notebook F3.

Por ello, en esta sección se ejecuta un pipeline equivalente al desarrollado en F2, reutilizando las funciones del módulo `src/funciones_reutilizables.py` cuando está disponible. El flujo incluye:

1. Normalización de nombres de columnas.
2. Conversión de porcentajes a formato numérico.
3. Transformación básica de fecha y hora cuando existen las columnas.
4. Creación de variables derivadas del resultado.
5. Imputación de valores nulos numéricos.
6. Codificación One-Hot de variables categóricas.
7. Escalamiento z-score de variables numéricas continuas.
8. Validación final: sin nulos, sin columnas no numéricas y con dimensiones válidas.

Se conserva un DataFrame algorítmico (`df`) para el cálculo interpretable de goles y se genera un DataFrame modelable (`df_pipeline_f2_completo`) que evidencia el pipeline completo solicitado.


In [4]:
def transformar_fecha_hora_segura(df: pd.DataFrame) -> pd.DataFrame:
    """Crea variables temporales derivadas si existen columnas de fecha y hora.

    La función es tolerante a formatos distintos: si alguna columna no existe,
    simplemente retorna el DataFrame sin fallar. Esto mantiene la ejecución
    reproducible en distintos entornos.
    """
    df = df.copy()

    if "date" in df.columns:
        fecha = pd.to_datetime(df["date"], errors="coerce")
        if fecha.notna().any():
            df["match_day"] = fecha.dt.day.fillna(0).astype(int)
            df["match_month"] = fecha.dt.month.fillna(0).astype(int)
        df = df.drop(columns=["date"])

    if "hour" in df.columns:
        hora = (
            df["hour"]
            .astype(str)
            .str.replace(" ", "", regex=False)
            .str.split(":")
            .str[0]
        )
        df["match_hour"] = pd.to_numeric(hora, errors="coerce").fillna(0).astype(int)
        df = df.drop(columns=["hour"])

    return df


def codificar_categoricas(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """Codifica variables categóricas con One-Hot Encoding.

    Decisión técnica: se aplica One-Hot Encoding porque las categorías son nominales
    y no tienen un orden natural. Esto evita imponer jerarquías artificiales.
    """
    df = df.copy()
    grupos_one_hot: dict[str, list[str]] = {}

    columnas_categoricas = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    for columna in columnas_categoricas:
        if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "codificar_one_hot_auto"):
            df, nuevas = f2_utils.codificar_one_hot_auto(df, columna)
        else:
            dummies = pd.get_dummies(df[columna], prefix=columna.replace(" ", "_"), dtype=int)
            nuevas = dummies.columns.tolist()
            df = pd.concat([df.drop(columns=[columna]), dummies], axis=1)

        grupos_one_hot[columna] = nuevas

    return df, grupos_one_hot


def columnas_continuas_para_escalar(df: pd.DataFrame, excluir: list[str] | None = None) -> list[str]:
    """Selecciona columnas numéricas continuas, excluyendo binarias y variables objetivo."""
    excluir = excluir or []
    columnas = []

    for col in df.select_dtypes(include=[np.number]).columns:
        if col in excluir:
            continue
        valores = set(pd.Series(df[col].dropna().unique()).astype(float).tolist())
        es_binaria = valores.issubset({0.0, 1.0})
        if not es_binaria:
            columnas.append(col)

    return columnas


def validar_pipeline_completo(df: pd.DataFrame) -> bool:
    """Valida que el pipeline completo deje datos consistentes para análisis/modelado."""
    total_nulos = int(df.isnull().sum().sum())
    no_numericas = df.select_dtypes(exclude=[np.number]).columns.tolist()

    assert df.shape[0] > 0, "El dataset final quedó sin filas."
    assert df.shape[1] > 0, "El dataset final quedó sin columnas."
    assert total_nulos == 0, f"Quedan {total_nulos} valores nulos."
    assert not no_numericas, f"Quedan columnas no numéricas: {no_numericas}"

    print("[OK] Pipeline completo validado")
    print(f"[OK] Filas: {df.shape[0]}")
    print(f"[OK] Columnas finales: {df.shape[1]}")
    print(f"[OK] Valores nulos: {total_nulos}")
    print("[OK] Todas las columnas finales son numéricas")
    return True


def ejecutar_pipeline_f2_completo(df_entrada: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, list[str]]]:
    """Ejecuta el pipeline completo reutilizado de F2 dentro del notebook F3.

    Retorna
    -------
    tuple
        DataFrame final modelable, reporte de etapas y grupos one-hot creados.
    """
    reporte = []
    df_work = df_entrada.copy()
    reporte.append(("entrada", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 1. Normalización de nombres de columnas.
    if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "normalizar_nombres_columnas"):
        df_work = f2_utils.normalizar_nombres_columnas(df_work)
    else:
        df_work = normalizar_nombres_columnas(df_work)
    reporte.append(("normalizacion_columnas", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 2. Casting de porcentajes.
    columnas_porcentaje = [
        "possession team1",
        "possession team2",
        "possession in contest"
    ]
    for col in columnas_porcentaje:
        if col in df_work.columns:
            if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "convertir_porcentaje"):
                df_work = f2_utils.convertir_porcentaje(df_work, col)
            else:
                df_work = convertir_porcentaje(df_work, col)
    reporte.append(("casting_porcentajes", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 3. Fecha/hora a variables numéricas derivadas.
    df_work = transformar_fecha_hora_segura(df_work)
    reporte.append(("transformacion_fecha_hora", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 4. Variables derivadas del resultado.
    if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "crear_variables_resultado"):
        df_work = f2_utils.crear_variables_resultado(df_work)
        # La versión del módulo F2 crea diferencia y resultado; F3 agrega total_goals para análisis.
        if "total_goals" not in df_work.columns:
            df_work["total_goals"] = df_work["number of goals team1"] + df_work["number of goals team2"]
    else:
        df_work = crear_variables_resultado(df_work)
    reporte.append(("variables_resultado", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 5. Imputación de nulos numéricos.
    numericas = df_work.select_dtypes(include=[np.number]).columns.tolist()
    for col in numericas:
        if df_work[col].isnull().any():
            if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "imputar_nulos_numericos"):
                df_work = f2_utils.imputar_nulos_numericos(df_work, col, estrategia="mediana")
            else:
                df_work[col] = df_work[col].fillna(df_work[col].median())
    reporte.append(("imputacion_numerica", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 6. Codificación de variables categóricas.
    df_work, grupos_one_hot = codificar_categoricas(df_work)
    reporte.append(("one_hot_categoricas", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 7. Escalamiento de variables numéricas continuas.
    excluir_escalamiento = [
        "number of goals team1",
        "number of goals team2",
        "total_goals",
        "goal_difference_team1",
    ]
    columnas_escalar = columnas_continuas_para_escalar(df_work, excluir=excluir_escalamiento)

    if columnas_escalar:
        if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "escalar_caracteristicas"):
            df_work, _ = f2_utils.escalar_caracteristicas(df_work, columnas_escalar)
        else:
            df_work = escalar_zscore(df_work, columnas_escalar)
    reporte.append(("escalamiento_zscore", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 8. Validación final.
    validar_pipeline_completo(df_work)
    reporte.append(("validacion_final", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    reporte_df = pd.DataFrame(
        reporte,
        columns=["etapa", "filas", "columnas", "valores_nulos"]
    )

    return df_work, reporte_df, grupos_one_hot


df_pipeline_f2_completo, reporte_pipeline_f2, grupos_one_hot_f2 = ejecutar_pipeline_f2_completo(df_original)

print("\nResumen de etapas del pipeline completo reutilizado de F2:")
display(reporte_pipeline_f2)

print("\nPrimeras columnas del dataset final modelable:")
display(df_pipeline_f2_completo.head())


[OK] Pipeline completo validado
[OK] Filas: 6
[OK] Columnas finales: 31
[OK] Valores nulos: 0
[OK] Todas las columnas finales son numéricas

Resumen de etapas del pipeline completo reutilizado de F2:


,etapa,filas,columnas,valores_nulos
0,entrada,6,12,0
1,normalizacion_columnas,6,12,0
2,casting_porcentajes,6,12,0
3,transformacion_fecha_hora,6,12,0
4,variables_resultado,6,15,0
5,imputacion_numerica,6,15,0
6,one_hot_categoricas,6,31,0
7,escalamiento_zscore,6,31,0
8,validacion_final,6,31,0



Primeras columnas del dataset final modelable:


,number of goals team1,number of goals team2,possession team1,possession team2,possession in contest,total attempts team1,total attempts team2,passes team1,passes team2,total_goals,goal_difference_team1,team1_ARGENTINA,team1_BRAZIL,team1_ENGLAND,team1_FRANCE,team1_QATAR,team1_SPAIN,team2_AUSTRALIA,team2_COSTA RICA,team2_ECUADOR,team2_IRAN,team2_SAUDI ARABIA,team2_SERBIA,category_Group A,category_Group B,category_Group C,category_Group D,category_Group E,category_Group G,result_team1_loss,result_team1_win
0,0,2,-1.646426,1.830705,0.426798,-1.710970,0.739940,-1.350242,1.242622,2,-2,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0
1,6,2,0.993533,-0.890159,-1.036508,-0.466628,1.547147,0.498483,-1.175700,8,4,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1
2,1,2,0.227093,-0.184750,-0.304855,-0.311086,-0.470871,-0.574628,-0.692035,3,-1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0
3,4,1,-0.283866,0.319114,0.060971,1.088799,-0.067267,0.158487,1.114861,5,3,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1
4,2,0,-0.624506,0.218341,1.890103,1.244342,-0.067267,-0.542753,0.539940,2,2,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1


### Interpretación del pipeline completo

El pipeline completo deja un dataset final sin valores nulos y con variables numéricas, después de aplicar casting, codificación categórica y escalamiento. Esta sección corrige explícitamente la observación del profesor: en F3 ya no se trabaja solo con la transformación de goles, sino que se ejecuta el flujo completo de preprocesamiento reutilizado desde F2 antes de desarrollar el núcleo algorítmico.

Para mantener interpretabilidad deportiva, el núcleo algorítmico de goles usa `df`, que conserva columnas como `team1` y `team2`; mientras que `df_pipeline_f2_completo` evidencia el dataset procesado completo para análisis/modelado.
